# Dhivehi OCR with Qwen 2.5 3B

This notebook performs OCR on Dhivehi PDFs using a fine-tuned Qwen 2.5 3B model hosted on Google Drive.

In [ ]:
# 1. Install Dependencies
!apt-get install -y poppler-utils
!pip install pdf2image transformers accelerate bitsandbytes sentencepiece

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Configuration and Model Loading
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pdf2image import convert_from_path
import os
from tqdm import tqdm

# Update these paths to match your Google Drive structure
MODEL_PATH = "/content/drive/MyDrive/models/qwen2.5-3b-ocr" 
INPUT_DIR = "/content/drive/MyDrive/mcq/data/saruna"
OUTPUT_DIR = "/content/drive/MyDrive/mcq/data/extracted_text"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    load_in_4bit=True # Optional: use 4-bit quantization if memory is an issue
)

In [ ]:
# 4. OCR Function
def perform_ocr(pdf_path):
    images = convert_from_path(pdf_path)
    full_text = ""
    
    for i, image in enumerate(images):
        # Note: Depending on your fine-tuned Qwen model's input requirements,
        # you might need to convert the image to a prompt or use a specific processor.
        # This is a generic causal LM inference placeholder.
        
        prompt = "Transcribe the following Dhivehi text from the image:"
        # Example: inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
        # For now, we assume the model takes text/image inputs or has been adapted.
        
        # If this is Qwen2-VL, use the VL-specific inference code here.
        
        # Simple text placeholder
        # inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        # out = model.generate(**inputs, max_new_tokens=1024)
        # page_text = tokenizer.decode(out[0], skip_special_tokens=True)
        # full_text += page_text + "\n"
        
        pass # Replace with actual VL model inference
        
    return full_text

In [ ]:
# 5. Main Processing Loop
pdf_files = []
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

for pdf_path in tqdm(pdf_files, desc="OCR Processing"):
    filename = os.path.basename(pdf_path)
    txt_path = os.path.join(OUTPUT_DIR, os.path.splitext(filename)[0] + ".txt")
    
    if os.path.exists(txt_path):
        continue
        
    text = perform_ocr(pdf_path)
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(text)